In [58]:
import h3
import pandas as pd
import numpy as np
from openai import OpenAI
import hdbscan
from sklearn.preprocessing import normalize
import umap

In [59]:
df_ramos = pd.read_excel('coordenadas_brasil.xlsx', sheet_name='RAMOS')
df_ramos

,RAMO,DESCRICAO,GRUPO
0,1,Desenvolvimento de software e sistemas digitais,Tecnologia
1,2,Consultoria em gestão e estratégia empresarial,Consultoria
2,3,Comércio varejista de produtos alimentícios,Varejo
3,4,Prestação de serviços de saúde e medicina,Saúde
4,5,Produção e distribuição de energia elétrica,Energia
5,6,Desenvolvimento de aplicativos móveis e plataf...,Tecnologia
6,7,Suporte técnico em infraestrutura de TI e redes,Tecnologia
7,8,Criação de soluções de inteligência artificial...,Tecnologia
8,9,Consultoria em finanças corporativas e fusões,Consultoria
9,10,Consultoria em recursos humanos e cultura orga...,Consultoria


In [60]:
api_key = 'xxx'
client = OpenAI(api_key=api_key)

def get_embedding(texto: str) -> list[float]:
    response = client.embeddings.create(
        input=texto,
        model="text-embedding-ada-002"
    )
    return response.data[0].embedding

df_ramos["EMBEDDING"] = df_ramos["DESCRICAO"].apply(get_embedding)
df_ramos

,RAMO,DESCRICAO,GRUPO,EMBEDDING
0,1,Desenvolvimento de software e sistemas digitais,Tecnologia,"[-0.009192856959998608, 0.00115637993440032, 0..."
1,2,Consultoria em gestão e estratégia empresarial,Consultoria,"[-0.007203449960798025, -0.01097150705754757, ..."
2,3,Comércio varejista de produtos alimentícios,Varejo,"[-0.0008728544926270843, -0.004586076363921165..."
3,4,Prestação de serviços de saúde e medicina,Saúde,"[0.00835702009499073, 0.01741831749677658, 0.0..."
4,5,Produção e distribuição de energia elétrica,Energia,"[-0.003879331052303314, -0.024461863562464714,..."
5,6,Desenvolvimento de aplicativos móveis e plataf...,Tecnologia,"[0.0009994972497224808, 0.0009517506114207208,..."
6,7,Suporte técnico em infraestrutura de TI e redes,Tecnologia,"[-0.006481224671006203, -0.010480956174433231,..."
7,8,Criação de soluções de inteligência artificial...,Tecnologia,"[-0.006067461334168911, 0.0060706427320837975,..."
8,9,Consultoria em finanças corporativas e fusões,Consultoria,"[-0.012391665019094944, -0.011882595717906952,..."
9,10,Consultoria em recursos humanos e cultura orga...,Consultoria,"[-0.014824132435023785, -0.014629589393734932,..."


In [61]:
X = normalize(np.array(df_ramos["EMBEDDING"].tolist()))

reducer = umap.UMAP(n_components=10, metric="cosine", random_state=42)
X_reduced = reducer.fit_transform(X)

# ── 4. HDBSCAN ────────────────────────────────────────────────────────────────
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,      # mínimo de pontos para formar um cluster
    min_samples=1,           # sensibilidade a ruído (1 = menos ruído)
    metric="euclidean",      # euclidiana em vetores normalizados ≈ coseno
    cluster_selection_method="eom"  # Excess of Mass: clusters mais estáveis
)

#df_ramos["CLUSTER_RAMO"] = clusterer.fit_predict(X)
df_ramos["CLUSTER_RAMO"] = clusterer.fit_predict(X_reduced)
df_ramos.loc[df_ramos["CLUSTER_RAMO"] < 0, 'CLUSTER_RAMO'] = None

df_ramos

C:\Users\lmene\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,RAMO,DESCRICAO,GRUPO,EMBEDDING,CLUSTER_RAMO
0,1,Desenvolvimento de software e sistemas digitais,Tecnologia,"[-0.009192856959998608, 0.00115637993440032, 0...",0.0
1,2,Consultoria em gestão e estratégia empresarial,Consultoria,"[-0.007203449960798025, -0.01097150705754757, ...",0.0
2,3,Comércio varejista de produtos alimentícios,Varejo,"[-0.0008728544926270843, -0.004586076363921165...",2.0
3,4,Prestação de serviços de saúde e medicina,Saúde,"[0.00835702009499073, 0.01741831749677658, 0.0...",3.0
4,5,Produção e distribuição de energia elétrica,Energia,"[-0.003879331052303314, -0.024461863562464714,...",1.0
5,6,Desenvolvimento de aplicativos móveis e plataf...,Tecnologia,"[0.0009994972497224808, 0.0009517506114207208,...",0.0
6,7,Suporte técnico em infraestrutura de TI e redes,Tecnologia,"[-0.006481224671006203, -0.010480956174433231,...",0.0
7,8,Criação de soluções de inteligência artificial...,Tecnologia,"[-0.006067461334168911, 0.0060706427320837975,...",0.0
8,9,Consultoria em finanças corporativas e fusões,Consultoria,"[-0.012391665019094944, -0.011882595717906952,...",0.0
9,10,Consultoria em recursos humanos e cultura orga...,Consultoria,"[-0.014824132435023785, -0.014629589393734932,...",0.0


In [62]:
df_depara_ramo = pd.concat([
    pd.DataFrame({
        'RAMO': df_ramos['RAMO'],
        'CLUSTER_RAMO_0': df_ramos['RAMO'],
        'CLUSTER_RAMO_1': df_ramos['CLUSTER_RAMO'],
        'CLUSTER_RAMO_2': 'ALL'
    }),
    pd.DataFrame({
        'RAMO': [0],
        'CLUSTER_RAMO_0': np.nan,
        'CLUSTER_RAMO_1': np.nan,
        'CLUSTER_RAMO_2': 'ALL',
    })
]).sort_values(by=['RAMO'], ascending=True).reset_index(drop=True)

df_depara_ramo

,RAMO,CLUSTER_RAMO_0,CLUSTER_RAMO_1,CLUSTER_RAMO_2
0,0,NaN,NaN,ALL
1,1,1.0,0.0,ALL
2,2,2.0,0.0,ALL
3,3,3.0,2.0,ALL
4,4,4.0,3.0,ALL
5,5,5.0,1.0,ALL
6,6,6.0,0.0,ALL
7,7,7.0,0.0,ALL
8,8,8.0,0.0,ALL
9,9,9.0,0.0,ALL


In [63]:
for i in range(0, 3):
    df_aux = pd.DataFrame({
        'RAMO': df_depara_ramo['RAMO'],
        'NIVEL_CLUSTER_RAMO': i,
        'CLUSTER_RAMO': df_depara_ramo[f'CLUSTER_RAMO_{i}']
    })

    df_aux = df_aux[df_aux['CLUSTER_RAMO'].isna()==False]

    if i==0:
        df_depara_ramo_lin = df_aux.copy().reset_index(drop=True)
    else:
        df_depara_ramo_lin = pd.concat([df_depara_ramo_lin, df_aux], ignore_index=True)

df_depara_ramo_lin

,RAMO,NIVEL_CLUSTER_RAMO,CLUSTER_RAMO
0,1,0,1.0
1,2,0,2.0
2,3,0,3.0
3,4,0,4.0
4,5,0,5.0
...,...,...,...
56,16,2,ALL
57,17,2,ALL
58,18,2,ALL
59,19,2,ALL


In [64]:
df_treino = pd.read_excel('coordenadas_brasil.xlsx', sheet_name='TREINO')
df_treino

,CNPJ,RAMO,LAT,LNG,FAT_DEB,FAT_ROT,FAT_PSJ
0,1,5,NaN,NaN,748,25,941
1,2,5,3.346872,-69.678734,459,218,993
2,3,3,-5.187596,-67.763425,305,401,170
3,4,2,-10.390346,-52.616689,170,982,70
4,5,5,NaN,NaN,860,810,328
...,...,...,...,...,...,...,...
49995,49996,5,-13.554424,-54.680244,891,371,501
49996,49997,14,-17.815964,-51.694321,700,703,734
49997,49998,16,-26.754078,-39.942176,805,784,453
49998,49999,16,-1.238190,-52.050550,408,997,501


In [65]:
df_treino = pd.merge(df_treino, df_depara_ramo, on='RAMO')

reg = 0
for res in range(9, -1, -1):
    df_treino[f'REG{reg}'] = df_treino.apply(
        lambda row: h3.latlng_to_cell(row['LAT'], row['LNG'], res) if pd.notna(row['LAT']) and pd.notna(row['LNG']) else None,
        axis=1
    )
    reg += 1

df_treino[f'REG{reg}'] = 'ALL'
df_treino

,CNPJ,RAMO,LAT,LNG,FAT_DEB,FAT_ROT,FAT_PSJ,CLUSTER_RAMO_0,CLUSTER_RAMO_1,CLUSTER_RAMO_2,...,REG1,REG2,REG3,REG4,REG5,REG6,REG7,REG8,REG9,REG10
0,1,5,NaN,NaN,748,25,941,5.0,1.0,ALL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ALL
1,2,5,3.346872,-69.678734,459,218,993,5.0,1.0,ALL,...,88665cd459fffff,87665cd45ffffff,86665cd47ffffff,85665cd7fffffff,84665cdffffffff,83665cfffffffff,82665ffffffffff,81667ffffffffff,8067fffffffffff,ALL
2,3,3,-5.187596,-67.763425,305,401,170,3.0,2.0,ALL,...,888a7024d7fffff,878a7024dffffff,868a7024fffffff,858a7027fffffff,848a715ffffffff,838a71fffffffff,828a77fffffffff,818a7ffffffffff,808bfffffffffff,ALL
3,4,2,-10.390346,-52.616689,170,982,70,2.0,0.0,ALL,...,88816c854bfffff,87816c854ffffff,86816c857ffffff,85816c87fffffff,84816c9ffffffff,83816cfffffffff,82816ffffffffff,81817ffffffffff,808bfffffffffff,ALL
4,5,5,NaN,NaN,860,810,328,5.0,1.0,ALL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ALL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,49996,5,-13.554424,-54.680244,891,371,501,5.0,1.0,ALL,...,888bb38cc3fffff,878bb38ccffffff,868bb38cfffffff,858bb38ffffffff,848bb39ffffffff,838bb3fffffffff,828bb7fffffffff,818bbffffffffff,808bfffffffffff,ALL
49996,49997,14,-17.815964,-51.694321,700,703,734,14.0,2.0,ALL,...,88a8ee8acbfffff,87a8ee8acffffff,86a8ee8afffffff,85a8ee8bfffffff,84a8ee9ffffffff,83a8eefffffffff,82a8effffffffff,81a8fffffffffff,80a9fffffffffff,ALL
49997,49998,16,-26.754078,-39.942176,805,784,453,16.0,3.0,ALL,...,88a9914c53fffff,87a9914c5ffffff,86a9914c7ffffff,85a9914ffffffff,84a9915ffffffff,83a991fffffffff,82a997fffffffff,81a9bffffffffff,80a9fffffffffff,ALL
49998,49999,16,-1.238190,-52.050550,408,997,501,16.0,3.0,ALL,...,88806899c7fffff,87806899cffffff,86806899fffffff,8580689bfffffff,8480689ffffffff,838068fffffffff,82806ffffffffff,81807ffffffffff,8081fffffffffff,ALL


In [66]:
df_cortes = pd.read_excel('coordenadas_brasil.xlsx', sheet_name='CORTES')

for i, row in df_cortes.iterrows():
    cluster_ramo = row['CLUSTER_RAMO']
    reg = row['REG']
    corte = row['CORTE']

    df_aux = pd.DataFrame({
        'NIVEL_CLUSTER_RAMO': cluster_ramo,
        'NIVEL_REG': reg,
        'CLUSTER_RAMO': df_treino[f'CLUSTER_RAMO_{cluster_ramo}'],
        'REG': df_treino[f'REG{reg}'],
        'QTD': 1,
        'FAT_DEB': df_treino['FAT_DEB'],
        'FAT_ROT': df_treino['FAT_ROT'],
        'FAT_PSJ': df_treino['FAT_PSJ'],
        'FAT_TT': df_treino['FAT_DEB'] + df_treino['FAT_ROT'] + df_treino['FAT_PSJ'],
    })

    df_aux = df_aux[(df_aux['CLUSTER_RAMO'].isna()==False)&(df_aux['REG'].isna()==False)]

    df_aux = df_aux.groupby(['NIVEL_CLUSTER_RAMO','NIVEL_REG','CLUSTER_RAMO','REG']).agg({
        'QTD': 'count',
        'FAT_DEB': 'sum',
        'FAT_ROT': 'sum',
        'FAT_PSJ': 'sum',
        'FAT_TT': 'sum'
    }).reset_index()

    df_aux = df_aux[df_aux['QTD']>=corte]

    if i==0:
        df_tt = df_aux.copy().reset_index(drop=True)
    else:
        df_tt = pd.concat([df_tt, df_aux], ignore_index=True)

df_tt['MIX_DEB'] = df_tt['FAT_DEB'] / df_tt['FAT_TT']
df_tt['MIX_ROT'] = df_tt['FAT_ROT'] / df_tt['FAT_TT']
df_tt['MIX_PSJ'] = df_tt['FAT_PSJ'] / df_tt['FAT_TT']

df_tt.drop(columns=['QTD','FAT_DEB','FAT_ROT','FAT_PSJ','FAT_TT'], inplace=True)
df_tt

,NIVEL_CLUSTER_RAMO,NIVEL_REG,CLUSTER_RAMO,REG,MIX_DEB,MIX_ROT,MIX_PSJ
0,0,0,1.0,8956d86c413ffff,0.154771,0.168404,0.676824
1,0,0,1.0,8956d955b73ffff,0.246146,0.405961,0.347893
2,0,0,1.0,8956d985d8fffff,0.500390,0.156128,0.343482
3,0,0,1.0,8956d98a52bffff,0.032723,0.948953,0.018325
4,0,0,1.0,8956d99934fffff,0.370213,0.236170,0.393617
...,...,...,...,...,...,...,...
674735,2,9,ALL,80a9fffffffffff,0.333739,0.334073,0.332187
674736,2,9,ALL,80b3fffffffffff,0.332091,0.331766,0.336143
674737,2,9,ALL,80c3fffffffffff,0.331472,0.341150,0.327379
674738,2,9,ALL,80c5fffffffffff,0.334090,0.325752,0.340158


In [67]:
df_tt = pd.merge(df_depara_ramo_lin, df_tt, on=['NIVEL_CLUSTER_RAMO','CLUSTER_RAMO'], how='inner').sort_values(by=['RAMO','NIVEL_REG','NIVEL_CLUSTER_RAMO']).reset_index(drop=True)
df_tt = df_tt.drop_duplicates(subset=['RAMO','NIVEL_REG','REG'], keep='first').reset_index(drop=True)
df_tt.drop(columns=['NIVEL_CLUSTER_RAMO','CLUSTER_RAMO'], inplace=True)
df_tt

,RAMO,NIVEL_REG,REG,MIX_DEB,MIX_ROT,MIX_PSJ
0,0,0,8956d80320fffff,0.491228,0.499452,0.009320
1,0,0,8956d8091a7ffff,0.895023,0.041629,0.063348
2,0,0,8956d814b63ffff,0.335160,0.342922,0.321918
3,0,0,8956d8166dbffff,0.472008,0.015444,0.512548
4,0,0,8956d818637ffff,0.193462,0.363189,0.443350
...,...,...,...,...,...,...
4155559,20,9,80a9fffffffffff,0.313791,0.347600,0.338609
4155560,20,9,80b3fffffffffff,0.350062,0.323521,0.326416
4155561,20,9,80c3fffffffffff,0.228375,0.498816,0.272809
4155562,20,9,80c5fffffffffff,0.376274,0.315218,0.308508


In [69]:
df_tt.to_csv('BASE_MODELO.csv', index=False)

In [75]:
df_modelo = pd.read_csv('BASE_MODELO.csv')
df_modelo.rename(columns={'RAMO': 'RAMO_BUSCADO'}, inplace=True)
df_modelo

,RAMO_BUSCADO,NIVEL_REG,REG,MIX_DEB,MIX_ROT,MIX_PSJ
0,0,0,8956d80320fffff,0.491228,0.499452,0.009320
1,0,0,8956d8091a7ffff,0.895023,0.041629,0.063348
2,0,0,8956d814b63ffff,0.335160,0.342922,0.321918
3,0,0,8956d8166dbffff,0.472008,0.015444,0.512548
4,0,0,8956d818637ffff,0.193462,0.363189,0.443350
...,...,...,...,...,...,...
4155559,20,9,80a9fffffffffff,0.313791,0.347600,0.338609
4155560,20,9,80b3fffffffffff,0.350062,0.323521,0.326416
4155561,20,9,80c3fffffffffff,0.228375,0.498816,0.272809
4155562,20,9,80c5fffffffffff,0.376274,0.315218,0.308508


In [76]:
df_teste = pd.read_excel('coordenadas_brasil.xlsx', sheet_name='TESTE')
df_teste

,CNPJ,RAMO,LAT,LNG
0,100001,18,-3.317569,-42.618899
1,100002,7,-16.643272,-41.946181
2,100003,7,NaN,NaN
3,100004,24,4.411151,-63.916923
4,100005,4,-12.737891,-34.794547
...,...,...,...,...
9995,109996,5,-9.184552,-61.256612
9996,109997,6,-5.419543,-59.337042
9997,109998,20,NaN,NaN
9998,109999,17,-29.496797,-44.517009


In [81]:
ramos_modelo = df_modelo['RAMO_BUSCADO'].drop_duplicates()

df_teste['RAMO_BUSCADO'] = df_teste['RAMO'].where(df_teste['RAMO'].isin(ramos_modelo), other=0)
df_teste

,CNPJ,RAMO,LAT,LNG,RAMO_BUSCADO
0,100001,18,-3.317569,-42.618899,18
1,100002,7,-16.643272,-41.946181,7
2,100003,7,NaN,NaN,7
3,100004,24,4.411151,-63.916923,0
4,100005,4,-12.737891,-34.794547,4
...,...,...,...,...,...
9995,109996,5,-9.184552,-61.256612,5
9996,109997,6,-5.419543,-59.337042,6
9997,109998,20,NaN,NaN,20
9998,109999,17,-29.496797,-44.517009,17


In [82]:
reg = 0
for res in range(10, -1, -1):
    df_aux = df_teste.copy()
    df_aux['NIVEL_REG'] = reg

    if res == 0:
        df_aux['REG'] = 'ALL'
    else:
        df_aux = df_aux[(df_aux['LAT'].isna()==False)&(df_aux['LNG'].isna()==False)].reset_index(drop=True)
        df_aux['REG'] = df_aux.apply(
            lambda row: h3.latlng_to_cell(row['LAT'], row['LNG'], res-1),
            axis=1
        )

    if reg==0:
        df_teste_2 = df_aux.copy()
    else:
        df_teste_2 = pd.concat([df_teste_2, df_aux], ignore_index=True)

    reg += 1

df_teste_2

,CNPJ,RAMO,LAT,LNG,RAMO_BUSCADO,NIVEL_REG,REG
0,100001,18,-3.317569,-42.618899,18,0,898003a4ccbffff
1,100002,7,-16.643272,-41.946181,7,0,898126d1c8fffff
2,100004,24,4.411151,-63.916923,0,0,895f650aec3ffff
3,100005,4,-12.737891,-34.794547,4,0,89a5a6922afffff
4,100006,12,-14.196280,-39.387434,12,0,89813220463ffff
...,...,...,...,...,...,...,...
89895,109996,5,-9.184552,-61.256612,5,10,ALL
89896,109997,6,-5.419543,-59.337042,6,10,ALL
89897,109998,20,NaN,NaN,20,10,ALL
89898,109999,17,-29.496797,-44.517009,17,10,ALL


In [86]:
df_teste_3 = pd.merge(df_teste_2, df_modelo, how='inner', on=['RAMO_BUSCADO','NIVEL_REG','REG'])
df_teste_3 = df_teste_3.sort_values(by=['CNPJ','NIVEL_REG']).drop_duplicates(subset='CNPJ', keep='first').reset_index(drop=True)
df_teste_3

,CNPJ,RAMO,LAT,LNG,RAMO_BUSCADO,NIVEL_REG,REG,MIX_DEB,MIX_ROT,MIX_PSJ
0,100001,18,-3.317569,-42.618899,18,4,858003a7fffffff,0.000000,0.476378,0.523622
1,100002,7,-16.643272,-41.946181,7,4,858126d3fffffff,0.380609,0.167612,0.451779
2,100003,7,NaN,NaN,7,10,ALL,0.336237,0.328387,0.335376
3,100004,24,4.411151,-63.916923,0,6,835f65fffffffff,0.307911,0.354489,0.337600
4,100005,4,-12.737891,-34.794547,4,5,84a5a69ffffffff,0.145833,0.484195,0.369971
...,...,...,...,...,...,...,...,...,...,...
9995,109996,5,-9.184552,-61.256612,5,5,848a339ffffffff,0.077165,0.127559,0.795276
9996,109997,6,-5.419543,-59.337042,6,4,858aac23fffffff,0.719643,0.125000,0.155357
9997,109998,20,NaN,NaN,20,10,ALL,0.330159,0.338717,0.331124
9998,109999,17,-29.496797,-44.517009,17,5,84a9a17ffffffff,0.382167,0.323192,0.294641
